### Preprocessing

This is for making the flight data readable for the NN
- Learn the FENNEC clipper (can it be improved?)
- Take data from the excel file and import into the NN

In [ ]:
#Imports
import os
import glob
import torch
import pandas as pd
import random
import numpy as np
from sklearn.preprocessing import StandardScaler
from torch.utils.data import TensorDataset, DataLoader

#What do each of these do?
# os: Provides functions to interact with the operating system.
# glob: Used to find all the pathnames matching a specified pattern.
# torch: PyTorch library for tensor computations and deep learning.
# pandas as pd: Library for data manipulation and analysis.
# random: Provides functions for generating random numbers.
# numpy as np: Library for numerical computations.
# StandardScaler: Standardizes features by removing the mean and scaling to unit variance.
# TensorDataset, DataLoader: Utilities for handling datasets and creating data loaders in PyTorch.


 ### What extract_cg_code does, step by step:

 Isolate the filename. os.path.basename removes any leading directory structure, so the function works regardless of whether filepath is absolute, relative, or just a bare filename.
 
 Strip the extension. os.path.splitext(filename)[0] removes anything after the last dot (.png, .dcm, .csv, etc.), leaving a clean base name to parse.
 
 Split on underscores. The filename is assumed to follow a convention like prefix_CODE_suffix.ext — e.g. scan_FST_001.dcm — where fields are delimited by _.
 
 Validate structure. If there's no second field (index 1), the filename doesn't match the expected convention, so it raises a clear error rather than crashing with an IndexError later.
 
 Extract and validate the code. parts[1] is assumed to be the 3-letter CG code (like "FST" from the earlier function). Its length is checked — exactly 3 characters — to catch malformed codes early, before they reach extract_axis_labels, where each individual character is validated against the axis maps.

In [ ]:
#Extracting code
#This is a def and a defintion can be called in another code to be used. 
#filepath is defined in the original code at the top where all of the terms are defined
def extract_cg_code(filepath):
    # Strip any directory path, leaving just the filename.
    # e.g. "/data/scans/img_FST_001.png" -> "img_FST_001.png"
    filename = os.path.basename(filepath)

    # Drop the extension, keeping only the name portion.
    # e.g. "img_FST_001.png" -> "img_FST_001"
    name = os.path.splitext(filename)[0]

    # Split the name on underscores to get its component fields.
    # e.g. "img_FST_001" -> ["img", "FST", "001"]
    parts = name.split("_") #THIS IS IMPORTANT FOR HOW WE STRUCTURE OUR DATA COLLECTION

    # The filename must have at least 2 underscore-separated parts,
    # since the CG code is expected to be the second one (index 1).
    if len(parts) < 2:
        raise ValueError(f"Invalid file format: {filename}")

    # Pull out the second part — this is assumed to be the CG code.
    cg_code = parts[1]

    # The CG code must be exactly 3 characters (one per axis: X, Y, Z),
    # matching what extract_axis_labels expects downstream.
    if len(cg_code) != 3:
        raise ValueError(f"Invalid CG code length: {filename}")

    return cg_code



### What extract_axis_labels does, step by step:

Get the code. extract_cg_code(filepath) (defined elsewhere) returns a short string like "FST" — presumably derived from the filename or file contents. "CG" hints this might stand for something like "center of gravity" or a similar three-axis classification scheme, though the exact domain depends on your codebase.

Split into three characters. code[0], code[1], code[2] assume the code is exactly 3 characters, one per axis (X, Y, Z).

Map letters to numbers. Each axis has its own small dictionary translating a letter into an index 0–2, essentially turning a human-readable label (like 'F' for "front") into a numeric position usable for indexing, sorting, or arithmetic elsewhere in the code.

Validate before mapping. Each character is checked against its map. If it's not a valid key, a ValueError is raised with a message identifying the axis, the bad character, and the file — which makes debugging much easier than a generic KeyError.

Return the numeric tuple. The final result is (x_index, y_index, z_index), e.g. ('F','S','T') → (0, 2, 0).

In [ ]:
#Extract Axis Label: This is for classification model
#I WILL NOT NEED THIS CODE, or just my labels will be different since they will be numerical
def extract_axis_labels(filepath):
    # Pull a 3-character code from the file (e.g. "FST").
    # Each character encodes a position along one spatial axis.
    code = extract_cg_code(filepath)   # example: FST
    x_char, y_char, z_char = code[0], code[1], code[2]

    # Lookup tables mapping each valid letter to a numeric index (0, 1, or 2).
    # These act like little enums: letter -> ordinal position on that axis.
    x_map = {'F': 0, 'M': 1, 'A': 2}   # X-axis options
    y_map = {'P': 0, 'C': 1, 'S': 2}   # Y-axis options
    z_map = {'T': 0, 'N': 1, 'B': 2}   # Z-axis options

    # Validate each character before using it, so a bad code fails loudly
    # and tells you exactly which axis and file caused the problem.
    if x_char not in x_map:
        raise ValueError(f"Invalid X label '{x_char}' in {filepath}")
    if y_char not in y_map:
        raise ValueError(f"Invalid Y label '{y_char}' in {filepath}")
    if z_char not in z_map:
        raise ValueError(f"Invalid Z label '{z_char}' in {filepath}")

    # Convert the three validated letters into a tuple of integer indices.
    return x_map[x_char], y_map[y_char], z_map[z_char]

### What it does, step by step:

Constructor (__init__) — runs automatically whenever you create a Flight object, e.g. f = Flight("data/scan_FST_001.csv"). It takes one input (filepath) and uses it to populate several pieces of derived information, all stored as attributes on self.

self.filepath / self.filename — two ways of referring to the file: the full path (for actually reading it) and just the name (handy for display or logs without leaking directory structure).

self.cg_code — reuses your earlier extract_cg_code function to pull the 3-letter code out of the filename.

self.label_x, self.label_y, self.label_z — reuses extract_axis_labels to convert that code into three numeric indices, one per axis, and unpacks them straight into three separate attributes.

self.data — calls the object's own load_data method (note self.load_data(), not a bare function call) to read the actual CSV contents into memory, storing the result on the object.

load_data method — uses pandas.read_csv to load the file into a DataFrame, then .values converts it to a plain NumPy array (dropping column names and the index, keeping just the raw numeric/data grid).

In [ ]:
class Flight:
    def __init__(self, filepath):
        # Store the original path so it's available later (e.g. for error
        # messages, re-loading, or logging which file this object came from)
        self.filepath = filepath  # full path to CSV

        # Extract just the filename (no directory) — useful for display,
        # logging, or comparisons without caring about where the file lives
        self.filename = os.path.basename(filepath)

        # Parse the 3-letter code (e.g. "FST") out of the filename,
        # using the function from before. Raises ValueError if the
        # filename doesn't follow the expected naming convention.
        self.cg_code = extract_cg_code(filepath)

        # Convert that code into numeric axis indices (x, y, z),
        # e.g. "FST" -> (0, 2, 0). Raises ValueError if any letter
        # isn't a recognized label for its axis.
        self.label_x, self.label_y, self.label_z = extract_axis_labels(filepath)

        # Load the actual flight data from the CSV and store it as
        # a NumPy array on the object
        self.data = self.load_data()

    def load_data(self):
        # load CSV, return as np.array - load vars of interest?
        df = pd.read_csv(self.filepath)
        return df.values